In [ ]:
# Cell 1: clone repo va sync dependency bang uv
import os
from pathlib import Path

%cd /content

REPO_URL = "https://github.com/vietanh-io/ai-video-content-management-system.git"
BRANCH_NAME = "ai-service/chaptering"  # doi thanh branch da push neu can
REPO_DIR = Path("/content/vid-pilot")

if not Path("/root/.local/bin/uv").exists():
    !curl -LsSf https://astral.sh/uv/install.sh | sh

os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

if not REPO_DIR.exists():
    !git clone --branch {BRANCH_NAME} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH_NAME}
    !git checkout {BRANCH_NAME}
    !git pull --ff-only origin {BRANCH_NAME}

%cd /content/vid-pilot/ai-service
!uv sync


In [ ]:
# Cell 2: config embedding + OpenAI-compatible LLM cho chaptering smoke.
# Secrets co the dat trong Colab: HF_TOKEN, LLM_BASE_URL, LLM_API_KEY, LLM_MODEL_NAME.
# Neu dung OpenAI truc tiep, co the chi can OPENAI_API_KEY; base URL default la https://api.openai.com/v1.
# Neu dung vLLM/SGLang local tunnel, dat LLM_BASE_URL thanh endpoint /v1 va LLM_MODEL_NAME thanh served model.
import os
from google.colab import userdata

def get_secret(name, default=""):
    try:
        return userdata.get(name) or default
    except Exception:
        return default

try:
    import torch

    DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    GPU_GB = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if torch.cuda.is_available() else 0
except Exception:
    DEFAULT_DEVICE = "cpu"
    GPU_NAME = "unknown"
    GPU_GB = 0

HF_TOKEN = get_secret("HF_TOKEN") or get_secret("HUGGING_FACE_HUB_TOKEN")
CACHE_DIR = "/content/hf_cache"

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = CACHE_DIR
os.environ["TOKENIZERS_PARALLELISM"] = "false"

os.environ["CHAPTERING_STRATEGY"] = "segment"  # doi thanh "word" neu transcript co word timestamps tot
os.environ["CHAPTERING_MODEL_NAME"] = "segment-chaptering-v1-smoke"
os.environ["CHAPTERING_EMBEDDING_PROVIDER"] = "sentence-transformers"
os.environ["CHAPTERING_EMBEDDING_MODEL_NAME"] = "Qwen/Qwen3-Embedding-0.6B"
os.environ["CHAPTERING_EMBEDDING_DEVICE"] = DEFAULT_DEVICE
os.environ["CHAPTERING_EMBEDDING_BATCH_SIZE"] = "4"
os.environ["CHAPTERING_EMBEDDING_MAX_SEQUENCE_LENGTH"] = "2048"
os.environ["CHAPTERING_EMBEDDING_CACHE_PATH"] = f"{CACHE_DIR}/sentence-transformers"
os.environ["CHAPTERING_EMBEDDING_LOCAL_FILES_ONLY"] = "false"

# LLM providers. De noop neu chi muon smoke embedding, nhung notebook nay mac dinh test ca title + boundary evaluate.
os.environ["CHAPTERING_BOUNDARY_EVALUATION_PROVIDER"] = "openai-compatible"
os.environ["CHAPTERING_TITLE_PROVIDER"] = "openai-compatible"
os.environ["LLM_BASE_URL"] = get_secret("LLM_BASE_URL", "https://api.openai.com/v1")
os.environ["LLM_API_KEY"] = get_secret("LLM_API_KEY") or get_secret("OPENAI_API_KEY")
os.environ["LLM_MODEL_NAME"] = get_secret("LLM_MODEL_NAME", "gpt-4o-mini")
os.environ["LLM_TIMEOUT_SECONDS"] = "90"
os.environ["LLM_TEMPERATURE"] = "0"
os.environ["LLM_MAX_TOKENS"] = "4096"

# Smoke options: nho de tranh prompt qua dai va khong dung de ket luan chat luong.
os.environ["CHAPTER_SMOKE_MIN_CHAPTER_SECONDS"] = "30"
os.environ["CHAPTER_SMOKE_TARGET_CHAPTER_SECONDS"] = "120"
os.environ["CHAPTER_SMOKE_MAX_CHAPTER_SECONDS"] = "360"
os.environ["CHAPTER_SMOKE_MAX_CHAPTERS"] = "6"
os.environ["CHAPTERING_CONTEXT_WINDOW_SECONDS"] = "15"
os.environ["CHAPTERING_EMBEDDING_CANDIDATE_MIN_LIMIT"] = "4"
os.environ["CHAPTERING_EMBEDDING_CANDIDATE_MAX_LIMIT"] = "6"
os.environ["CHAPTERING_EMBEDDING_CANDIDATE_MULTIPLIER"] = "1"

print("HF token:", "co" if HF_TOKEN else "chua co")
print("GPU:", GPU_NAME, f"{GPU_GB}GB")
print("Embedding device:", os.environ["CHAPTERING_EMBEDDING_DEVICE"])
print("Embedding model:", os.environ["CHAPTERING_EMBEDDING_MODEL_NAME"])
print("LLM base URL:", os.environ["LLM_BASE_URL"])
print("LLM model:", os.environ["LLM_MODEL_NAME"])
print("LLM key:", "co" if os.environ["LLM_API_KEY"] else "khong co / local endpoint")


In [ ]:
# Cell 3: chon transcript JSON trong Google Drive.
# Goi y: copy output JSON tu transcript_smoke_colab.ipynb vao MyDrive/chaptering_smoke/transcripts/.
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/chaptering_smoke")
TRANSCRIPT_DIR = BASE_DIR / "transcripts"
RESULT_DIR = BASE_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TRANSCRIPT_FILENAME = "your_transcript.json"  # sua ten file o day
TRANSCRIPT_PATH = TRANSCRIPT_DIR / TRANSCRIPT_FILENAME
RESULT_PATH = RESULT_DIR / f"{Path(TRANSCRIPT_FILENAME).stem}_chaptering_smoke.json"
REPORT_PATH = RESULT_DIR / f"{Path(TRANSCRIPT_FILENAME).stem}_chaptering_smoke_report.json"

assert TRANSCRIPT_PATH.exists(), f"Khong thay file: {TRANSCRIPT_PATH}"
print("Transcript:", TRANSCRIPT_PATH)
print("Result:", RESULT_PATH)
print("Report:", REPORT_PATH)


In [ ]:
# Cell 4: download NLTK data needed by lexical scoring.
import nltk

NLTK_PACKAGES = [
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "wordnet",
    "omw-1.4",
]

for package in NLTK_PACKAGES:
    nltk.download(package, quiet=False)

print("NLTK data ready:", ", ".join(NLTK_PACKAGES))


In [ ]:
# Cell 5: smoke embedding + LLM boundary evaluate + LLM title generation.
import json
import os
import subprocess
from pathlib import Path

RUNNER = Path("/content/vid-pilot/ai-service/chaptering_smoke_runner.py")
RUNNER.write_text(
    r'''
import json
import os
import sys
from pathlib import Path

from app.core.config import get_settings
from app.runtime.container import (
    _build_chaptering_pipeline_config,
    build_chaptering_boundary_evaluation_provider,
    build_chaptering_embedding_provider,
    build_chaptering_title_provider,
)
from app.schemas.chaptering import ChapterGenerationRequest, ChapteringOptions
from app.scripts.chaptering.buid_unit import (
    _load_segments,
    _media_duration,
    _text_value,
)
from app.workflows.chaptering.candidate_ranking import prepare_boundary_candidates_for_review
from app.workflows.chaptering.common import media_duration
from app.workflows.chaptering.llm_evaluation import build_boundary_evaluation_inputs
from app.workflows.chaptering.schemas import BoundaryEvaluationInput, ChapterTitleInput
from app.workflows.chaptering.scores.gap_scoring import (
    attach_semantic_shift_scores,
    gap_scores_to_candidates,
    score_unit_gaps,
)
from app.workflows.chaptering.scores.valleys import detect_valley_candidates
from app.workflows.chaptering.segment_units import build_segment_chapter_units
from app.workflows.chaptering.semantic import score_context_windows
from app.workflows.chaptering.title_generation import build_chapter_title_inputs
from app.workflows.chaptering.unit_repair import repair_micro_units
from app.workflows.chaptering.windows import build_context_windows
from app.workflows.chaptering.workflow import ChapteringWorkflow

transcript_path = Path(sys.argv[1])
result_path = Path(sys.argv[2])
report_path = Path(sys.argv[3])

payload = json.loads(transcript_path.read_text(encoding="utf-8"))
if not isinstance(payload, dict):
    raise SystemExit("Transcript JSON must contain an object payload.")

segments = _load_segments(payload, synthesize_ids=True)
media_duration_seconds = _media_duration(payload, segments)
language = _text_value(payload, "language")
if not language and isinstance(payload.get("transcript"), dict):
    language = _text_value(payload["transcript"], "language")

settings = get_settings()
config = _build_chaptering_pipeline_config(settings)
embedding = build_chaptering_embedding_provider(settings)
boundary_provider = build_chaptering_boundary_evaluation_provider(settings)
title_provider = build_chaptering_title_provider(settings)

if settings.chaptering_boundary_evaluation_provider != "openai-compatible":
    raise RuntimeError("Boundary evaluation provider must be openai-compatible for this LLM smoke.")
if settings.chaptering_title_provider != "openai-compatible":
    raise RuntimeError("Title provider must be openai-compatible for this LLM smoke.")

options = ChapteringOptions(
    min_chapter_duration_seconds=float(os.environ.get("CHAPTER_SMOKE_MIN_CHAPTER_SECONDS", "30")),
    target_chapter_duration_seconds=float(os.environ.get("CHAPTER_SMOKE_TARGET_CHAPTER_SECONDS", "120")),
    max_chapter_duration_seconds=float(os.environ.get("CHAPTER_SMOKE_MAX_CHAPTER_SECONDS", "360")),
    max_chapters=int(os.environ.get("CHAPTER_SMOKE_MAX_CHAPTERS", "6")),
    use_embeddings=True,
    use_llm=True,
)
request = ChapterGenerationRequest(
    request_id=f"colab-chaptering-smoke-{transcript_path.stem}",
    language=language or None,
    media_duration_seconds=media_duration_seconds,
    segments=segments,
    options=options,
)

direct_embeddings = embedding.embed_texts(
    [
        "This section explains uploading long-form media and storing metadata.",
        "This section explains chapter boundary scoring and transcript analysis.",
    ]
)
if len(direct_embeddings) != 2 or not direct_embeddings[0]:
    raise RuntimeError("Embedding provider did not return valid smoke vectors.")

direct_boundary = boundary_provider.evaluate_boundaries(
    [
        BoundaryEvaluationInput(
            candidate_time=12.5,
            left_context="The creator introduces the upload workflow and input media.",
            right_context="The next section moves into transcript analysis and chapter scoring.",
            candidate_score=0.72,
            lexical_shift_score=0.64,
            semantic_shift_score=0.71,
            valley_depth_score=0.55,
            pause_score=0.2,
            discourse_marker_score=0.5,
        )
    ]
)
if len(direct_boundary) != 1 or direct_boundary[0].candidate_time != 12.5:
    raise RuntimeError("Boundary LLM smoke returned an unknown candidate time.")

direct_titles = title_provider.generate_titles(
    [
        ChapterTitleInput(
            chapter_index=1,
            start_time=0,
            end_time=60,
            language=language or "en",
            text="Upload media, generate transcripts, score chapter boundaries, and review chapter labels.",
        )
    ]
)
if len(direct_titles) != 1 or direct_titles[0].chapter_index != 1 or not direct_titles[0].title.strip():
    raise RuntimeError("Title LLM smoke did not return a valid title for the provided chapter index.")

units = build_segment_chapter_units(
    request.segments,
    max_unit_duration=config.max_unit_duration_seconds,
    pause_boundary_seconds=config.pause_boundary_seconds,
    target_unit_duration=config.target_unit_duration_seconds,
    target_unit_words=config.target_unit_words,
    max_unit_words=config.max_unit_words,
    max_unit_chars=config.max_unit_chars,
    punctuation_poor_threshold=config.punctuation_poor_threshold,
)
units = repair_micro_units(units, pause_boundary_seconds=config.pause_boundary_seconds, config=config.unit_repair)
duration = media_duration(request)
gap_scores = score_unit_gaps(
    units,
    media_duration=duration,
    min_chapter_duration=options.min_chapter_duration_seconds,
    config=config.scoring,
)
all_gap_candidates = gap_scores_to_candidates(gap_scores)
all_gap_windows = build_context_windows(units, all_gap_candidates, context_duration=config.context_window_seconds)
semantic_shift_scores_by_time = score_context_windows(all_gap_windows, embedding=embedding) if all_gap_windows else {}
gap_scores = attach_semantic_shift_scores(gap_scores, semantic_shift_scores_by_time)
valley_gap_scores = detect_valley_candidates(
    gap_scores,
    min_candidate_distance_seconds=options.min_chapter_duration_seconds / 2,
    config=config.valley,
)
valleys_by_time = {gap_score.time: gap_score for gap_score in valley_gap_scores}
candidate_gap_scores = (
    [valleys_by_time.get(gap_score.time, gap_score) for gap_score in gap_scores]
    if valley_gap_scores
    else gap_scores
)
scored_candidates = gap_scores_to_candidates(candidate_gap_scores)
review_candidates = prepare_boundary_candidates_for_review(
    scored_candidates,
    max_chapters=options.max_chapters,
    min_candidate_distance_seconds=options.min_chapter_duration_seconds / 2,
    config=config.retention,
)
review_windows = build_context_windows(units, review_candidates, context_duration=config.context_window_seconds)
boundary_inputs = build_boundary_evaluation_inputs(review_candidates, review_windows)
review_candidate_times = sorted(round(candidate.time, 3) for candidate in review_candidates)
candidate_debug = [
    {
        "time": round(candidate.time, 3),
        "unitIndex": candidate.unit_index,
        "candidateScore": candidate.candidate_score,
        "semanticShiftScore": candidate.semantic_shift_score,
        "lexicalShiftScore": candidate.lexical_shift_score,
        "valleyDepthScore": candidate.valley_depth_score,
        "pauseScore": candidate.pause_score,
        "discourseMarkerScore": candidate.discourse_marker_score,
        "boundaryQualityScore": candidate.boundary_quality_score,
    }
    for candidate in review_candidates
]
debug_base = {
    "transcriptPath": str(transcript_path),
    "inputSegments": len(segments),
    "mediaDurationSeconds": media_duration_seconds,
    "effectiveDurationSeconds": duration,
    "options": options.model_dump(),
    "config": {
        "targetUnitDurationSeconds": config.target_unit_duration_seconds,
        "maxUnitDurationSeconds": config.max_unit_duration_seconds,
        "contextWindowSeconds": config.context_window_seconds,
        "retentionMinLimit": config.retention.min_limit,
        "retentionMaxLimit": config.retention.max_limit,
        "retentionMultiplier": config.retention.multiplier,
        "valleyMinDepth": config.valley.min_valley_depth,
    },
    "unitCount": len(units),
    "gapScoreCount": len(gap_scores),
    "allGapCandidateCount": len(all_gap_candidates),
    "valleyGapScoreCount": len(valley_gap_scores),
    "scoredCandidateCount": len(scored_candidates),
    "reviewCandidateCount": len(review_candidate_times),
    "boundaryInputCount": len(boundary_inputs),
    "reviewCandidateTimes": review_candidate_times,
    "reviewCandidates": candidate_debug,
}

if not review_candidate_times:
    report_path.write_text(json.dumps({**debug_base, "failure": "no_review_candidates"}, ensure_ascii=False, indent=2), encoding="utf-8")
    raise RuntimeError("No review candidates were produced. Use a longer transcript or lower min chapter duration.")

workflow = ChapteringWorkflow(
    embedding=embedding,
    boundary_evaluator=boundary_provider,
    title_provider=title_provider,
    config=config,
)
result = workflow.execute(request)
data = result.model_dump()
chapters = data["chapters"]
selected_starts = [round(chapter["start_seconds"], 3) for chapter in chapters]

if data["source"] != "LLM":
    raise RuntimeError("Pipeline source is not LLM; title or boundary provider probably fell back.")
if len(chapters) < 2:
    data["smoke"] = {**debug_base, "failure": "fewer_than_2_chapters", "selectedStarts": selected_starts, "chapterCount": len(chapters)}
    result_path.parent.mkdir(parents=True, exist_ok=True)
    result_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    report_path.write_text(json.dumps(data["smoke"], ensure_ascii=False, indent=2), encoding="utf-8")
    print("Debug report:", report_path)
    print("Duration:", duration)
    print("Options:", options.model_dump())
    print("Units/gaps/review candidates:", len(units), len(gap_scores), len(review_candidates))
    print("Review candidate times:", review_candidate_times[:30])
    print("Selected starts:", selected_starts)
    raise RuntimeError("Smoke produced fewer than 2 chapters, so selected boundary LLM metadata cannot be checked.")

allowed_starts = {0.0, *review_candidate_times}
unknown_starts = [start for start in selected_starts if start not in allowed_starts]
if unknown_starts:
    raise RuntimeError(f"Pipeline selected starts outside candidate set: {unknown_starts}")

selected_boundary_scores = [
    (chapter.get("scores") or {}).get("llm_confidence_score") or 0
    for chapter in chapters
    if round(chapter["start_seconds"], 3) != 0.0
]
if not any(score > 0 for score in selected_boundary_scores):
    raise RuntimeError("No selected non-zero boundary has LLM confidence metadata.")

empty_titles = [chapter["index"] for chapter in chapters if not chapter["title"].strip()]
if empty_titles:
    raise RuntimeError(f"Generated chapters contain empty titles: {empty_titles}")

final_title_inputs = build_chapter_title_inputs(result.chapters, request)
final_title_smoke = title_provider.generate_titles(final_title_inputs)
final_title_indexes = sorted(item.chapter_index for item in final_title_smoke)
expected_title_indexes = sorted(chapter["index"] for chapter in chapters)
if final_title_indexes != expected_title_indexes:
    raise RuntimeError(
        f"Title LLM did not return exactly the final chapter indexes: {final_title_indexes} vs {expected_title_indexes}"
    )

semantic_scores = [
    (chapter.get("scores") or {}).get("semantic_shift_score")
    for chapter in chapters
    if chapter.get("scores")
]
report = {
    "transcriptPath": str(transcript_path),
    "inputSegments": len(segments),
    "mediaDurationSeconds": media_duration_seconds,
    "embeddingProvider": settings.chaptering_embedding_provider,
    "embeddingModel": embedding.model_name,
    "embeddingDimension": embedding.dimension,
    "directEmbeddingCount": len(direct_embeddings),
    "pipelineUsedEmbeddings": any(score is not None and score > 0 for score in semantic_scores),
    "llmBaseUrl": settings.llm_base_url,
    "llmModel": settings.llm_model_name,
    "directBoundarySmoke": {
        "candidateTime": direct_boundary[0].candidate_time,
        "isChapterBoundary": direct_boundary[0].is_chapter_boundary,
        "confidence": direct_boundary[0].confidence,
        "transitionIntent": direct_boundary[0].transition_intent,
    },
    "directTitleSmoke": {
        "chapterIndex": direct_titles[0].chapter_index,
        "title": direct_titles[0].title,
        "summary": direct_titles[0].summary,
    },
    "reviewCandidateCount": len(review_candidate_times),
    "boundaryInputCount": len(boundary_inputs),
    "reviewCandidateTimes": review_candidate_times,
    "selectedStarts": selected_starts,
    "selectedStartsAllFromCandidates": not unknown_starts,
    "selectedBoundaryLlmConfidenceScores": selected_boundary_scores,
    "finalTitleSmokeCount": len(final_title_smoke),
    "finalTitleSmokeIndexes": final_title_indexes,
}
data["smoke"] = report

result_path.parent.mkdir(parents=True, exist_ok=True)
result_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", result_path)
print("Report:", report_path)
print("Model:", data["model"])
print("Source:", data["source"])
print("Embedding model:", report["embeddingModel"])
print("Embedding dimension:", report["embeddingDimension"])
print("LLM model:", report["llmModel"])
print("Review candidates:", report["reviewCandidateCount"])
print("Selected starts all from candidates:", report["selectedStartsAllFromCandidates"])
print("Selected boundary LLM confidence scores:", report["selectedBoundaryLlmConfidenceScores"])
print("Direct title:", report["directTitleSmoke"]["title"])
print("Chapters:", len(chapters))
for chapter in chapters:
    scores = chapter.get("scores") or {}
    print(
        f"- {chapter['index']}: {chapter['start_seconds']:.2f}s -> "
        f"{chapter['end_seconds']:.2f}s | {chapter['title']} | "
        f"llm_conf={scores.get('llm_confidence_score')} "
        f"semantic={scores.get('semantic_shift_score')} "
        f"lexical={scores.get('lexical_shift_score')}"
    )
''',
    encoding="utf-8",
)

env = os.environ.copy()
completed = subprocess.run(
    ["uv", "run", "python", str(RUNNER), str(TRANSCRIPT_PATH), str(RESULT_PATH), str(REPORT_PATH)],
    cwd="/content/vid-pilot/ai-service",
    env=env,
    text=True,
    capture_output=True,
)

print(completed.stdout)
if completed.stderr:
    print("STDERR:")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(f"Chaptering LLM smoke failed with exit code {completed.returncode}")
